# 🛡️ NSL-KDD 이상탐지 → RAG → Qwen → HTML 위협보고서

**교육 목표**: 앞에서 배운 RAG 실습의 가상 이벤트를 **실제 NSL-KDD 샘플 CSV의 이상탐지 결과**로 바꿉니다.

**전체 흐름**: CSV 읽기 → 특징 선택 → Isolation Forest → 이상 이벤트 TOP 5 → RAG 검색(TF-IDF·코사인 유사도) → Qwen 보고서 생성 → HTML 다운로드

✅ OpenAI API Key 불필요 · ✅ `tokenizer()`와 `llm.generate()` 사용 · ✅ 가능한 한 간단한 코드

**준비**: Colab 왼쪽 파일 탭에 `Exercise3. nsl_kdd_sample.csv` 업로드. **T4 GPU** 권장.

> 데이터에는 원본 IP 주소나 이벤트 발생 시각이 없습니다. 임의의 IP·날짜를 만들지 않습니다. `label`과 `binary_label`은 학습이나 RAG 검색에 넣지 않고 마지막에 참고 정답 비교로만 사용합니다. 이상탐지 ≠ 공격 유형 판정입니다.

## 1. 설치 / 라이브러리

In [1]:
!pip -q install scikit-learn pandas numpy transformers accelerate torch

In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import HTML, display
from html import escape
from pathlib import Path

## 2. NSL-KDD CSV 불러오기

CSV 파일을 업로드한 다음 `pd.read_csv()`로 읽습니다. `label`, `binary_label`은 **정답 비교용**이고 모델 입력에 넣지 않아요.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
df = pd.read_csv("/content/drive/MyDrive/coc_class-main/ML/data/Exercise3. nsl_kdd_sample.csv")
print("데이터 크기:", df.shape)
display(df.head(3))

데이터 크기: (3000, 44)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty,binary_label
0,0,tcp,bgp,S0,0,0,0,0,0,0,...,0.08,0.00,0.00,1.0,1.0,0.0,0.00,neptune,19,attack
1,0,tcp,ftp_data,SF,201,0,0,0,0,0,...,0.76,0.18,0.03,0.0,0.0,0.7,0.01,normal,21,normal
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.07,0.00,0.00,1.0,1.0,0.0,0.00,neptune,21,attack


## 3. 이상탐지에 사용할 특성 8개 선택

숫자 정보 8개만 사용합니다. `StandardScaler`는 각 특성의 값 범위를 표준화합니다.

In [6]:
features = ["duration", "src_bytes", "dst_bytes", "count",
            "srv_count", "serror_rate", "dst_host_count", "num_failed_logins"]
X = df[features]
X_scaled = StandardScaler().fit_transform(X)
print("모델 입력 크기:", X_scaled.shape)

모델 입력 크기: (3000, 8)


## 4. Isolation Forest로 비지도 이상탐지

`fit_predict()`는 1(정상 쪽), -1(이상 쪽)을 출력합니다. `anomaly_score`는 **높을수록 이례적**인 값으로 변환했어요. `contamination=0.05`는 약 5%를 이상으로 선택하는 모델 설정이지 실제 공격 비율이 아닙니다.

In [7]:
detector = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
df["prediction"] = detector.fit_predict(X_scaled)
df["anomaly_score"] = -detector.score_samples(X_scaled)
df["event_id"] = [f"NSL-{i:04d}" for i in range(1, len(df)+1)]
print(df["prediction"].value_counts().rename(index={1: "정상 예측", -1: "이상 예측"}))

prediction
정상 예측    2850
이상 예측     150
Name: count, dtype: int64


## 5. 이상 점수가 높은 5건 뽑기

In [8]:
top5 = (df[df["prediction"] == -1]
        .sort_values("anomaly_score", ascending=False)
        .head(5).copy())
display(top5[["event_id", "protocol_type", "service", "src_bytes",
              "dst_bytes", "count", "serror_rate", "anomaly_score"]])

,event_id,protocol_type,service,src_bytes,dst_bytes,count,serror_rate,anomaly_score
1374,NSL-1375,tcp,IRC,44788,48103,1,1.0,0.710131
2215,NSL-2216,tcp,telnet,3555,198172,1,0.0,0.700783
514,NSL-0515,tcp,IRC,37681,40396,1,1.0,0.690060
1391,NSL-1392,tcp,telnet,794,152110,1,0.0,0.686802
1870,NSL-1871,tcp,telnet,4113,51633,1,0.0,0.665130


## 6. 보안 지식베이스 준비

실습용 대응 자료입니다. 각 문서는 **관찰할 특징, 추가 점검 항목, 대응 조치**를 담고 있습니다. 실제 데이터의 라벨을 미리 알려주지 않습니다.

In [9]:
docs = [
    {"title":"서비스 거부(DoS) 점검", "text":"count srv_count serror_rate dst_host_count 동시 연결과 SYN 오류가 많으면 서비스 거부 가능성을 점검. 서비스 응답률, 트래픽 급증, 방화벽 로그 확인. 대응: 속도 제한과 가용성 모니터링."},
    {"title":"스캔·정찰 점검", "text":"service protocol_type dst_host_count count 여러 대상이나 서비스 접근이 늘어나면 정찰 가능성 점검. 포트 접근 분포, IDS 로그, 후속 접속 확인. 대응: 노출 포트와 접근제어 점검."},
    {"title":"무차별 로그인 시도 점검", "text":"num_failed_logins logged_in 인증 실패가 많으면 계정 공격 가능성 점검. 인증 시각, 계정 목록, MFA 사용 여부 확인. 대응: 인증 로그 조사와 계정 보호."},
    {"title":"비정상 데이터 전송 점검", "text":"src_bytes dst_bytes duration 장시간 또는 대량 데이터 송수신이면 유출 가능성 점검. 목적지, 파일 전송 이력, 프록시/DLP 기록 확인. 대응: 전송 경로 분석."},
    {"title":"네트워크 이상 일반 점검", "text":"duration src_bytes dst_bytes count srv_count serror_rate dst_host_count 이상탐지 수치만으로 공격을 확정할 수 없음. 원본 로그, 정상 기준선, 관련 호스트 현황 확인."},
]
kb = pd.DataFrame(docs)
display(kb)

,title,text
0,서비스 거부(DoS) 점검,count srv_count serror_rate dst_host_count 동시 ...
1,스캔·정찰 점검,service protocol_type dst_host_count count 여러 ...
2,무차별 로그인 시도 점검,num_failed_logins logged_in 인증 실패가 많으면 계정 공격 가...
3,비정상 데이터 전송 점검,src_bytes dst_bytes duration 장시간 또는 대량 데이터 송수신...
4,네트워크 이상 일반 점검,duration src_bytes dst_bytes count srv_count s...


## 7. 이벤트를 검색용 문장으로 변경

우리가 확인한 **관측 값만** 문장으로 만듭니다. 공격 라벨은 사용하지 않아요.

In [10]:
def build_query(row):
    return (f"duration {row['duration']} src_bytes {row['src_bytes']} "
            f"dst_bytes {row['dst_bytes']} count {row['count']} "
            f"srv_count {row['srv_count']} serror_rate {row['serror_rate']} "
            f"dst_host_count {row['dst_host_count']} "
            f"num_failed_logins {row['num_failed_logins']} "
            f"protocol_type {row['protocol_type']} service {row['service']}")

print(build_query(top5.iloc[0]))

duration 0 src_bytes 44788 dst_bytes 48103 count 1 srv_count 1 serror_rate 1.0 dst_host_count 5 num_failed_logins 0 protocol_type tcp service IRC


## 8. RAG 검색: TF-IDF → 코사인 유사도 → TOP 2

문서와 이벤트를 숫자로 변환하고 가장 유사한 자료를 2개 가져옵니다. **유사도는 공격 발생 확률이 아닙니다.**

In [11]:
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(kb["title"] + " " + kb["text"])

def search_documents(query, top_k=2):
    q_vector = vectorizer.transform([query])
    scores = cosine_similarity(q_vector, doc_vectors)[0]
    result = kb.copy()
    result["similarity"] = scores
    return result.sort_values("similarity", ascending=False).head(top_k)

display(search_documents(build_query(top5.iloc[0]))[["title", "similarity"]])

,title,similarity
4,네트워크 이상 일반 점검,0.356877
1,스캔·정찰 점검,0.217088


## 9. Qwen LLM 불러오기

앞에서 배운 `AutoTokenizer`, `tokenizer()`, `llm.generate()`, `tokenizer.decode()`를 그대로 이용합니다.

> GPU가 없는 경우 CPU에서도 실행할 수 있지만 생성 속도가 느릴 수 있어요. 강의 시간에는 `max_new_tokens`를 180 정도로 제한하세요.

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
llm = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
llm.eval()
print("모델 실행 장치:", llm.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

## 10. LLM 답변 함수 만들기

채팅 지시를 잘 이해하도록 대화 형식 문자열을 먼저 만들고, **그 문자열을 `tokenizer()`로 토큰화**합니다. `llm.generate(**inputs)`로 앞서 본 `'shape'` 오류를 피합니다.

In [ ]:
def ask_llm(prompt, max_new_tokens=180):
    messages = [{"role":"user", "content":prompt}]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        output = llm.generate(**inputs, max_new_tokens=max_new_tokens,
                              do_sample=False, pad_token_id=tokenizer.eos_token_id)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 11. 5개 이벤트에 RAG + Qwen 연결

각 이벤트를 검색하고, 찾아낸 보안 문서 2개를 Qwen에게 함께 전달합니다. LLM에는 공격 정답을 주지 않아요.

**생성 시간 절약:** 먼저 1건만 시험하려면 `top5.head(1)`로 바꿔 실행해 보고, 성공하면 `top5.iterrows()`로 돌아오세요.

In [ ]:
reports = []
for _, row in top5.iterrows():
    found = search_documents(build_query(row), top_k=2)
    references = "\n".join(f"- {d['title']}: {d['text']}" for _, d in found.iterrows())

    prompt = f"""당신은 보안 분석가입니다. 교육용 NSL-KDD 네트워크 이상 이벤트를 한국어로 분석하세요.
이벤트 ID: {row['event_id']}
관측 값: {build_query(row)}
이상 점수: {row['anomaly_score']:.3f} (높을수록 이례적)
관련 보안 문서:
{references}

아래 4개 항목으로 1~2문장씩 짧게 작성하세요.
1. 이벤트 개요
2. 관측 근거 및 가능한 위협
3. 확인할 로그와 초기 대응
4. 분석 한계
주의: 문서는 참고자료이며 해당 공격을 확정하지 마세요. IP나 시각은 데이터에 없으므로 만들지 마세요."""

    analysis = ask_llm(prompt, max_new_tokens=200)
    reports.append({"event_id":row['event_id'], "score":float(row['anomaly_score']),
                    "observations":build_query(row),
                    "references":found[["title","similarity"]].to_dict("records"),
                    "analysis":analysis})
    print(f"✅ {row['event_id']} 보고서 생성 완료")

print("총 보고서:", len(reports), "건")
print(reports[0]["analysis"])

## 12. HTML 위협보고서로 모으기

**보고서 본문은 Qwen이 직접 작성**하고 Python은 5건의 내용을 한 HTML 파일에 정리해요. 0.5B 소형 모델이 긴 HTML 전체를 생성할 때 코드가 잘리는 문제를 예방하는 방식입니다.

아래의 HTML은 *표시 형식*만 정의합니다. 위협 설명은 하드코딩하지 않습니다.

In [ ]:

from IPython.display import HTML, display
from html import escape
from datetime import datetime
from pathlib import Path

# ✅ 1. HTML에 들어갈 텍스트 처리
safe = lambda value: escape(str(value))

# ✅ 2. 이상 이벤트 5건을 HTML 카드로 변환
cards = ""

for r in reports:

    # RAG 검색 문서 목록 만들기
    references = ""
    for doc in r["references"]:
        references += f"""
        <p>
            {safe(doc['title'])}
            (유사도: {doc['similarity']:.3f})
        </p>
        """

    # 이벤트별 보고서 카드 생성
    cards += f"""
    <section class="section">

        <h2>이벤트 {safe(r['event_id'])}</h2>
        <span class="tag">이상 징후 · 추가 분석 필요</span>

        <h3>01. 이벤트 개요</h3>
        <div class="field">
            <strong>이상탐지 점수:</strong>
            {r['score']:.3f}
        </div>

        <h3>02. 주요 관측 정보</h3>
        <pre>{safe(r['observations'])}</pre>

        <h3>03. RAG 검색 결과</h3>
        <div class="field">
            {references}
        </div>

        <h3>04. Qwen 위협 분석 결과</h3>
        <pre>{safe(r['analysis'])}</pre>

        <div class="warning">
            본 이벤트는 이상탐지 결과이며,
            실제 공격 여부는 추가 확인이 필요합니다.
        </div>

    </section>
    """

# ✅ 3. 최종 HTML 보고서 생성
html_report = f"""<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">

<title>NSL-KDD RAG 위협 분석 보고서</title>

<style>
* {{
    box-sizing: border-box;
}}

body {{
    margin: 0;
    font-family: 'Malgun Gothic', Arial, sans-serif;
    background: #f1f5f9;
    color: #17253a;
    line-height: 1.7;
}}

main {{
    max-width: 1100px;
    margin: 32px auto;
    padding: 0 18px 36px;
}}

header {{
    background: #142841;
    color: white;
    border-radius: 16px;
    padding: 34px 36px;
}}

header h1 {{
    margin: 0 0 10px;
    font-size: 28px;
}}

header p {{
    color: #bfd2e4;
    margin: 5px 0;
}}

.section {{
    margin-top: 20px;
    background: white;
    padding: 28px;
    border-radius: 14px;
    box-shadow: 0 2px 12px #182b430c;
}}

h2 {{
    color: #163658;
    margin-top: 0;
}}

h3 {{
    color: #28547d;
    font-size: 17px;
    margin-top: 25px;
}}

.field {{
    background: #f6f9fd;
    border: 1px solid #e6edf5;
    border-radius: 9px;
    padding: 15px;
}}

.tag {{
    display: inline-block;
    background: #e7effa;
    color: #15518e;
    padding: 5px 12px;
    border-radius: 99px;
    font-size: 12px;
    font-weight: bold;
}}

pre {{
    white-space: pre-wrap;
    overflow-wrap: anywhere;
    background: #f6f9fd;
    padding: 18px;
    border-radius: 10px;
    font: 14px/1.8 'Malgun Gothic', Arial, sans-serif;
}}

.warning {{
    margin-top: 20px;
    border-left: 4px solid #d99d34;
    padding: 14px;
    background: #fff8ea;
    border-radius: 5px;
}}

.summary {{
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 15px;
}}

.summary-box {{
    background: #f6f9fd;
    padding: 20px;
    border-radius: 10px;
    text-align: center;
}}

.summary-number {{
    font-size: 28px;
    font-weight: bold;
    color: #163658;
}}

footer {{
    margin-top: 30px;
    text-align: center;
    color: #60758a;
    font-size: 12px;
}}

@media print {{
    body {{ background: white; }}
    .section {{
        box-shadow: none;
        break-inside: avoid;
    }}
}}
</style>
</head>

<body>
<main>

<header>
    <h1>🛡️ NSL-KDD 위협 분석 보고서</h1>
    <p>Isolation Forest + RAG + Qwen LLM</p>
    <p>생성 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
</header>

<section class="section">
    <h2>종합 분석 현황</h2>

    <div class="summary">
        <div class="summary-box">
            <div>전체 분석 데이터</div>
            <div class="summary-number">{len(df):,}</div>
        </div>

        <div class="summary-box">
            <div>이상탐지 이벤트</div>
            <div class="summary-number">
                {(df['prediction'] == -1).sum():,}
            </div>
        </div>

        <div class="summary-box">
            <div>상세 분석 대상</div>
            <div class="summary-number">{len(reports)}</div>
        </div>
    </div>
</section>

{cards}

<section class="section">
    <h2>종합 결론 및 분석 한계</h2>
    <p>
        본 보고서는 NSL-KDD 데이터를 대상으로
        Isolation Forest 이상탐지 모델을 적용하고,
        탐지된 주요 이벤트를 RAG 및 Qwen LLM으로
        분석한 교육용 보고서입니다.
    </p>
    <p>
        이상탐지 점수와 검색된 문서의 유사도만으로
        실제 침해사고 여부 또는 공격 유형을 확정할 수 없습니다.
        추가적인 로그 분석 및 전문가 검증이 필요합니다.
    </p>
</section>

<footer>
    Generated by Isolation Forest + TF-IDF RAG + Qwen LLM
</footer>

</main>
</body>
</html>"""

# ✅ 4. HTML 파일 저장
file_name = "NSL_KDD_RAG_위협보고서.html"

Path(file_name).write_text(
    html_report,
    encoding="utf-8"
)

print("✅ HTML 보고서 생성 완료:", file_name)

# ✅ 5. 화면에 HTML 보고서 출력
display(HTML(html_report))


## 13. HTML 파일 다운로드

In [ ]:
from google.colab import files
files.download("NSL_KDD_RAG_위협보고서.html")